In [ ]:
import pandas as pd
import numpy as np
import pickle
from tqdm import tqdm
import statsmodels.api as sm
from tensorflow.keras.models import Sequential  
from tensorflow.keras.layers import Dense  
from tensorflow.keras.optimizers import Adam  
from tensorflow.keras.callbacks import EarlyStopping

In [ ]:
with open("data/raw/characteristics_with_return.pkl", "rb") as f:
    data = pickle.load(f)

In [ ]:
data.head()

In [ ]:
def R2_cal(y, y_pred):
    r2 = 1 - np.sum((y-y_pred)**2) / np.sum(y**2)
    return r2

In [ ]:
#决策树模型
# 60 years of data
from sklearn.tree import DecisionTreeRegressor

start_year_train = 1957
start_year_test = 1987
end_year_test = 2016

# Convert the 'Date' column to datetime if it's not already
data['DATE'] = pd.to_datetime(data['DATE'], format='%Y%m')

pred_y_list = []
true_y_list = []

# Loop over the years
for year in tqdm(range(start_year_test, end_year_test+1), desc='Processing', unit='item'):
    # Define the start and end dates for training and test
    start_date_train = pd.Timestamp(year=start_year_train, month=1, day=1)
    end_date_train = pd.Timestamp(year=year-13, month=12, day=31)

    start_date_val = pd.Timestamp(year=year-12, month=1, day=1)
    end_date_val = pd.Timestamp(year=year-1, month=12, day=31)
    
    start_date_test = pd.Timestamp(year=year, month=1, day=1)
    end_date_test = pd.Timestamp(year=year, month=12, day=31)
    

    print(f'start_date_train: {start_date_train.strftime("%Y-%m")}')
    print(f'end_date_train: {end_date_train.strftime("%Y-%m")}')

    print(f'start_date_val: {start_date_val.strftime("%Y-%m")}')
    print(f'end_date_val: {end_date_val.strftime("%Y-%m")}')
    
    print(f'start_date_test: {start_date_test.strftime("%Y-%m")}')
    print(f'end_date_test: {end_date_test.strftime("%Y-%m")}')

    # Select the data for training and test
    train_data = data[(data['DATE'] >= start_date_train) & (data['DATE'] <= end_date_val)]
    test_data = data[(data['DATE'] >= start_date_test) & (data['DATE'] <= end_date_test)]
    
    # Extract features and target variables
    X_train = train_data.drop(['permno', 'DATE', 'RET'], axis=1).values  
    y_train = train_data['RET'].values  
    X_test = test_data.drop(['permno', 'DATE', 'RET'], axis=1).values
    y_test = test_data['RET'].values
  
    model = DecisionTreeRegressor()  
    model.fit(X_train, y_train)   
  
    y_test_pred = model.predict(X_test)  

    pred_y_list.extend(y_test_pred)  
    true_y_list.extend(y_test)

In [ ]:
R2_cal(np.array(true_y_list), np.array(pred_y_list))* 100

In [ ]:
#单层神经网络
def relu(x):  
    return np.maximum(0, x)  
  
def relu_grad(x):  
    return (x > 0).astype(np.float32)

def mean_squared_error(y, t):
    return 0.5 * np.sum((y - t) ** 2)

class OneLayerNet:

    def __init__(self, input_size, output_size, weight_init_std=0.01):
        # 初始化权重
        self.params = {}
        self.params['W1'] = weight_init_std * np.random.randn(input_size, output_size)
        self.params['b1'] = np.zeros(output_size)

    def predict(self, x):
        W1 = self.params['W1']
        b1 = self.params['b1']
    
        a1 = np.dot(x, W1) + b1
        y = a1
        
        return y
        
    # x:输入数据, t:监督数据
    def loss(self, x, t):
        y = self.predict(x)
        return mean_squared_error(y, t)
    
    def accuracy(self, x, t):
        y = self.predict(x)
        error = np.mean((y - t) ** 2)  
        return error
        
    def gradient(self, x, t):
        W1 = self.params['W1']
        b1 = self.params['b1']
        grads = {}

        # 前向传播
        a1 = np.dot(x, W1) + b1
        y = relu(a1)  

        # 反向传播
        dy = (y - t) / x.shape[0]  
        da1 = dy * relu_grad(a1) 
        grads['W1'] = np.dot(x.T, da1) / x.shape[0] 
        grads['b1'] = np.sum(da1, axis=0) / x.shape[0]  

        return grads

In [ ]:
# 60 years of data
start_year_train = 1957
start_year_test = 1987
end_year_test = 2016

# Convert the 'Date' column to datetime if it's not already
data['DATE'] = pd.to_datetime(data['DATE'], format='%Y%m')
pred_y_list = []
true_y_list = []

# Loop over the years
for year in tqdm(range(start_year_test, end_year_test+1), desc='Processing', unit='item'):
    # Define the start and end dates for training and test
    start_date_train = pd.Timestamp(year=start_year_train, month=1, day=1)
    end_date_train = pd.Timestamp(year=year-13, month=12, day=31)

    start_date_val = pd.Timestamp(year=year-12, month=1, day=1)
    end_date_val = pd.Timestamp(year=year-1, month=12, day=31)
    
    start_date_test = pd.Timestamp(year=year, month=1, day=1)
    end_date_test = pd.Timestamp(year=year, month=12, day=31)
    

    print(f'start_date_train: {start_date_train.strftime("%Y-%m")}')
    print(f'end_date_train: {end_date_train.strftime("%Y-%m")}')

    print(f'start_date_val: {start_date_val.strftime("%Y-%m")}')
    print(f'end_date_val: {end_date_val.strftime("%Y-%m")}')
    
    print(f'start_date_test: {start_date_test.strftime("%Y-%m")}')
    print(f'end_date_test: {end_date_test.strftime("%Y-%m")}')

    # Select the data for training and test
    train_data = data[(data['DATE'] >= start_date_train) & (data['DATE'] <= end_date_train)]
    val_data = data[(data['DATE'] >= start_date_val) & (data['DATE'] <= end_date_val)]
    test_data = data[(data['DATE'] >= start_date_test) & (data['DATE'] <= end_date_test)]
    
    # Extract features and target variables
    X_train = train_data.drop(['permno', 'DATE', 'RET'], axis=1).values  
    y_train = train_data['RET'].values.flatten()  
    X_val = val_data.drop(['permno', 'DATE', 'RET'], axis=1).values 
    y_val = val_data['RET'].values.flatten()
    X_test = test_data.drop(['permno', 'DATE', 'RET'], axis=1).values
    y_test = test_data['RET'].values.flatten()
    
    network = OneLayerNet(input_size = 94, output_size = 1)  
 
    n_epochs = 100000   
    train_size = X_train.shape[0]
    batch_size = 100
    learning_rate = 0.01  

    patience = 20
    best_model_state = None
    best_loss = float('inf')
    counter = 0

# 神经网络训练循环
for epoch in range(n_epochs): 
    for i in range(0, train_size, batch_size):
        xi = X_train[i:i + batch_size]
        yi = y_train[i:i + batch_size]  
        grad = network.gradient(xi, yi)
        for key in ('W1', 'b1'):
            network.params[key] -= learning_rate * grad[key]

    # 使用验证集评估模型性能
    val_loss = network.accuracy(X_val, y_val)
    
    # 早停机制
    if val_loss < best_loss:
        best_loss = val_loss
        best_model_state = network.params.copy()
        counter = 0
    else:
        counter += 1
        if counter >= patience:
            break

    if epoch % 2000 == 0:
        train_loss = network.accuracy(X_train, y_train)
        print(f"Epoch {epoch}: train loss {train_loss}, val loss {val_loss}")

# 应用最佳模型参数
network.params = best_model_state

# 使用最佳模型进行预测
y_test_pred = network.predict(X_test)

# 将预测结果和真实结果添加到列表中
pred_y_list.extend(y_test_pred)
true_y_list.extend(y_test)
        
for key in ('W1', 'b1'):
    # 确保grad[key]的形状与network.params[key]的形状相同
    network.params[key] -= learning_rate * grad[key].squeeze()
        
# Predict on training and validation data
y_test_pred = model.predict(X_test)

pred_y_list.extend(y_test_pred)
true_y_list.extend(y_test)

In [ ]:
R2_cal(np.array(true_y_list), np.array(pred_y_list))* 100

In [ ]:
#双层神经网络
def relu(x):
    return np.maximum(0, x)

def relu_grad(x):
    return (x > 0).astype(np.float32)


class TwoLayerNet:

    def __init__(self, input_size, hidden_size, output_size, weight_init_std=0.01):
        # 初始化权重
        self.params = {}
        self.params['W1'] = weight_init_std * np.random.randn(input_size, hidden_size)
        self.params['b1'] = np.zeros(hidden_size)
        self.params['W2'] = weight_init_std * np.random.randn(hidden_size, output_size)
        self.params['b2'] = np.zeros(output_size)

    def predict(self, x):
        W1, W2 = self.params['W1'], self.params['W2']
        b1, b2 = self.params['b1'], self.params['b2']
    
        a1 = np.dot(x, W1) + b1
        z1 = relu(a1)
        a2 = np.dot(z1, W2) + b2
        y = a2  
        
        return y
        
    def loss(self, x, t):
        y = self.predict(x)
        return mean_squared_error(y, t)
    
    def accuracy(self, x, t):
        y = self.predict(x)
        error_sum = 0.0
        batch_size = 1000  # 您可以根据可用内存调整这个大小
        for i in range(0, x.shape[0], batch_size):
            batch_y = y[i:i + batch_size]
            batch_t = t[i:i + batch_size]
            error_sum += np.sum((batch_y - batch_t) ** 2)
        error = error_sum / x.shape[0]
        return error

    def gradient(self, x, t):
        W1, W2 = self.params['W1'], self.params['W2']
        b1, b2 = self.params['b1'], self.params['b2']
        grads = {}
    
        batch_size = x.shape[0]
    
        # 前向传播
        a1 = np.dot(x, W1) + b1
        z1 = relu(a1)
        a2 = np.dot(z1, W2) + b2
        y = a2
    
        # 反向传播
        # 确保 t 是列向量
        if t.ndim == 1:
            t = t[:, np.newaxis]
    
        # 计算 dy，其形状应为 (batch_size, output_size)
        # 这里 output_size 应该是 1，因为是回归问题
        dy = (y - t) / batch_size
    
        # 计算 da2，它是 dy 与权重 W2 的转置矩阵的乘积
        da2 = np.dot(dy, W2.T)
    
        # 计算 dz1，它是 da2 与 ReLU 激活函数的导数的逐元素乘积
        dz1 = da2 * relu_grad(z1)
    
        # 计算 W1 和 b1 的梯度
        grads['W1'] = np.dot(x.T, dz1) / batch_size
        grads['b1'] = np.sum(dz1, axis=0) / batch_size
    
        # 计算 W2 和 b2 的梯度
        grads['W2'] = np.dot(z1.T, dy) / batch_size
        grads['b2'] = np.sum(dy, axis=0) / batch_size
    
        return grads

In [ ]:
# 60 years of data
start_year_train = 1957
start_year_test = 1987
end_year_test = 2016

# Convert the 'Date' column to datetime if it's not already
data['DATE'] = pd.to_datetime(data['DATE'], format='%Y%m')
pred_y_list = []
true_y_list = []

# Loop over the years
for year in tqdm(range(start_year_test, end_year_test+1), desc='Processing', unit='item'):
    # Define the start and end dates for training and test
    start_date_train = pd.Timestamp(year=start_year_train, month=1, day=1)
    end_date_train = pd.Timestamp(year=year-13, month=12, day=31)

    start_date_val = pd.Timestamp(year=year-12, month=1, day=1)
    end_date_val = pd.Timestamp(year=year-1, month=12, day=31)
    
    start_date_test = pd.Timestamp(year=year, month=1, day=1)
    end_date_test = pd.Timestamp(year=year, month=12, day=31)
    

    print(f'start_date_train: {start_date_train.strftime("%Y-%m")}')
    print(f'end_date_train: {end_date_train.strftime("%Y-%m")}')

    print(f'start_date_val: {start_date_val.strftime("%Y-%m")}')
    print(f'end_date_val: {end_date_val.strftime("%Y-%m")}')
    
    print(f'start_date_test: {start_date_test.strftime("%Y-%m")}')
    print(f'end_date_test: {end_date_test.strftime("%Y-%m")}')

    # Select the data for training and test
    train_data = data[(data['DATE'] >= start_date_train) & (data['DATE'] <= end_date_train)]
    val_data = data[(data['DATE'] >= start_date_val) & (data['DATE'] <= end_date_val)]
    test_data = data[(data['DATE'] >= start_date_test) & (data['DATE'] <= end_date_test)]
    
    # Extract features and target variables
    X_train = train_data.drop(['permno', 'DATE', 'RET'], axis=1).values  
    y_train = train_data['RET'].values  
    X_val = val_data.drop(['permno', 'DATE', 'RET'], axis=1).values 
    y_val = val_data['RET'].values
    X_test = test_data.drop(['permno', 'DATE', 'RET'], axis=1).values
    y_test = test_data['RET'].values
    
    network = TwoLayerNet(input_size = 94, hidden_size = 32, output_size = 1)  
 
    n_epochs = 100000   
    train_size = X_train.shape[0]
    batch_size = 1000 
    learning_rate = 0.01  

    patience = 20
    best_model_state = None
    best_loss = float('inf')
    counter = 0

    for epoch in range(n_epochs): 
        for i in range(0, train_size, batch_size):
            xi = X_train[i:i + batch_size]
            yi = y_train[i:i + batch_size]  
        
            grad = network.gradient(xi, yi)  
            for key in ('W1', 'b1', 'W2', 'b2'):
                network.params[key] -= learning_rate * grad[key]
    
        val_loss = network.accuracy(X_val, y_val)
    
        if val_loss < best_loss:
            best_loss = val_loss
            best_model_state = network.params.copy()
            counter = 0
        else:
            counter += 1
            if counter >= patience:
                break

        if epoch % 2000 == 0:
            train_loss = network.accuracy(X_train, y_train)
            test_loss = network.accuracy(X_val, y_val)
            print(f"train loss: {train_loss}, val loss {val_loss}") 
        
    for key in ('W1', 'b1', 'W2', 'b2'):
        network.params[key] = best_model_state[key]
        
    # Predict on training and validation data
    y_test_pred = network.predict(X_test)  

    # 将预测结果和真实结果添加到列表中
    pred_y_list.extend(y_test_pred)
    true_y_list.extend(y_test)

In [ ]:
R2_cal(np.array(true_y_list), np.array(pred_y_list))* 100

In [ ]:
# 三层神经网络
class ThreeLayerNet:  
    def __init__(self, input_size, hidden_size1, hidden_size2, output_size, weight_init_std=0.01):  
        # 初始化权重  
        self.params = {}  
        self.params['W1'] = weight_init_std * np.random.randn(input_size, hidden_size1)  
        self.params['b1'] = np.zeros(hidden_size1)  
        self.params['W2'] = weight_init_std * np.random.randn(hidden_size1, hidden_size2)  
        self.params['b2'] = np.zeros(hidden_size2)  
        self.params['W3'] = weight_init_std * np.random.randn(hidden_size2, output_size)  
        self.params['b3'] = np.zeros(output_size)  
  
    def predict(self, x):   
        W1, W2, W3 = self.params['W1'], self.params['W2'], self.params['W3']  
        b1, b2, b3 = self.params['b1'], self.params['b2'], self.params['b3']  
          
        a1 = np.dot(x, W1) + b1  
        z1 = relu(a1)  
        a2 = np.dot(z1, W2) + b2  
        z2 = relu(a2)  
        a3 = np.dot(z2, W3) + b3  
        y = a3  
          
        return y  
  
    def loss(self, x, t):  
        y = self.predict(x)  
        return mean_squared_error(y, t)  

    def accuracy(self, x, t):
        y = self.predict(x)
        error = np.mean((y - t) ** 2)  
        
        return error
        
    def gradient(self, x, t):  
        grads = {}  
        W1, W2, W3 = self.params['W1'], self.params['W2'], self.params['W3']  
        b1, b2, b3 = self.params['b1'], self.params['b2'], self.params['b3']  
          
        batch_num = x.shape[0]  
          
        a1 = np.dot(x, W1) + b1  
        z1 = relu(a1)  
        a2 = np.dot(z1, W2) + b2  
        z2 = relu(a2)  
        a3 = np.dot(z2, W3) + b3  
        y = a3  
          
        dy = (y - t) / batch_num  
        grads['W3'] = np.dot(z2.T, dy)  
        grads['b3'] = np.sum(dy, axis=0)  
          
        dz2 = relu_grad(a2) * np.dot(dy, W3.T)  
        grads['W2'] = np.dot(z1.T, dz2)  
        grads['b2'] = np.sum(dz2, axis=0)  
          
        dz1 = relu_grad(a1) * np.dot(dz2, W2.T)  
        grads['W1'] = np.dot(x.T, dz1)  
        grads['b1'] = np.sum(dz1, axis=0)  
          
        return grads

In [ ]:
# 60 years of data
start_year_train = 1957
start_year_test = 1987
end_year_test = 2016

# Convert the 'Date' column to datetime if it's not already
data['DATE'] = pd.to_datetime(data['DATE'], format='%Y%m')
pred_y_list = []
true_y_list = []

# Loop over the years
for year in tqdm(range(start_year_test, end_year_test+1), desc='Processing', unit='item'):
    # Define the start and end dates for training and test
    start_date_train = pd.Timestamp(year=start_year_train, month=1, day=1)
    end_date_train = pd.Timestamp(year=year-13, month=12, day=31)

    start_date_val = pd.Timestamp(year=year-12, month=1, day=1)
    end_date_val = pd.Timestamp(year=year-1, month=12, day=31)
    
    start_date_test = pd.Timestamp(year=year, month=1, day=1)
    end_date_test = pd.Timestamp(year=year, month=12, day=31)
    

    print(f'start_date_train: {start_date_train.strftime("%Y-%m")}')
    print(f'end_date_train: {end_date_train.strftime("%Y-%m")}')

    print(f'start_date_val: {start_date_val.strftime("%Y-%m")}')
    print(f'end_date_val: {end_date_val.strftime("%Y-%m")}')
    
    print(f'start_date_test: {start_date_test.strftime("%Y-%m")}')
    print(f'end_date_test: {end_date_test.strftime("%Y-%m")}')

    # Select the data for training and test
    train_data = data[(data['DATE'] >= start_date_train) & (data['DATE'] <= end_date_train)]
    val_data = data[(data['DATE'] >= start_date_val) & (data['DATE'] <= end_date_val)]
    test_data = data[(data['DATE'] >= start_date_test) & (data['DATE'] <= end_date_test)]
    
    # Extract features and target variables
    X_train = train_data.drop(['permno', 'DATE', 'RET'], axis=1).values  
    y_train = train_data['RET'].values  
    X_val = val_data.drop(['permno', 'DATE', 'RET'], axis=1).values 
    y_val = val_data['RET'].values
    X_test = test_data.drop(['permno', 'DATE', 'RET'], axis=1).values
    y_test = test_data['RET'].values
    
    network = ThreeLayerNet(input_size = 94, hidden_size1 = 32, hidden_size2 = 16, output_size = 1)  
 
    n_epochs = 100000   
    train_size = x_train.shape[0]
    batch_size = 1000 
    learning_rate = 0.01  

    patience = 20
    best_model_state = None
    best_loss = float('inf')
    counter = 0

    for epoch in range(n_epochs): 
        for i in range(0, train_size, batch_size):
            xi = x_train[i:i + batch_size]
            yi = y_train[i:i + batch_size]  
        
            grad = net_work.gradient(xi, yi)
            for key in ('W1', 'b1', 'W2', 'b2', 'W3', 'b3'):
                net_work.params[key] -= learning_rate * grad[key]
    
        val_loss = net_work.accuracy(x_val, y_val)
    
        if val_loss < best_loss:
            best_loss = val_loss
            best_model_state = net_work.params.copy()
            counter = 0
        else:
            counter += 1
            if counter >= patience:
                break

        if epoch % 2000 == 0:
            train_loss = net_work.accuracy(x_train, y_train)
            test_loss = net_work.accuracy(x_val, y_val)
            print(f"train loss: {train_loss}, val loss {val_loss}") 
        
    for key in ('W1', 'b1', 'W2', 'b2', 'W3', 'b3'):
        net_work.params[key] = best_model_state[key]
        
    # Predict on training and validation data
    y_test_pred = model.predict(X_test)

    pred_y_list.extend(y_test_pred)
    true_y_list.extend(y_test)

In [ ]:
R2_cal(np.array(true_y_list), np.array(pred_y_list))* 100

In [ ]:
#四层神经网络  
class FourLayerNet:  
  
    def __init__(self, input_size, hidden_size1, hidden_size2, hidden_size3, output_size, weight_init_std=0.01):  
        # 初始化权重和偏置  
        self.params = {}  
        self.params['W1'] = weight_init_std * np.random.randn(input_size, hidden_size1)  
        self.params['b1'] = np.zeros(hidden_size1)  
        self.params['W2'] = weight_init_std * np.random.randn(hidden_size1, hidden_size2)  
        self.params['b2'] = np.zeros(hidden_size2)  
        self.params['W3'] = weight_init_std * np.random.randn(hidden_size2, hidden_size3)  
        self.params['b3'] = np.zeros(hidden_size3)  
        self.params['W4'] = weight_init_std * np.random.randn(hidden_size3, output_size)  
        self.params['b4'] = np.zeros(output_size)  
  
    def predict(self, x):  
        W1, W2, W3, W4 = self.params['W1'], self.params['W2'], self.params['W3'], self.params['W4']  
        b1, b2, b3, b4 = self.params['b1'], self.params['b2'], self.params['b3'], self.params['b4']  
          
        a1 = np.dot(x, W1) + b1  
        z1 = relu(a1)  
        a2 = np.dot(z1, W2) + b2  
        z2 = relu(a2)  
        a3 = np.dot(z2, W3) + b3  
        z3 = relu(a3) 
        a4 = np.dot(z3, W4) + b4  
        y = a4  
          
        return y  
  
    def loss(self, x, t):  
        y = self.predict(x)  
        return mean_squared_error(y, t)  

    def accuracy(self, x, t):
        y = self.predict(x)
        error = np.mean((y - t) ** 2)  
        
        return error
  
    def gradient(self, x, t):  
        grads = {}  
        W1, W2, W3, W4 = self.params['W1'], self.params['W2'], self.params['W3'], self.params['W4']
        b1, b2, b3, b4 = self.params['b1'], self.params['b2'], self.params['b3'], self.params['b4']
          
        batch_num = x.shape[0]  
          
        a1 = np.dot(x, W1) + b1  
        z1 = relu(a1)  
        a2 = np.dot(z1, W2) + b2  
        z2 = relu(a2)  
        a3 = np.dot(z2, W3) + b3 
        z3 = relu(a3)
        a4 = np.dot(z3, W4) + b4
        y = a4
          
        dy = (y - t) / batch_num  
        grads['W4'] = np.dot(z3.T, dy)  
        grads['b4'] = np.sum(dy, axis=0)  
          
        dz3 = relu_grad(a3) * np.dot(dy, W4.T)  
        grads['W3'] = np.dot(z2.T, dz3)  
        grads['b3'] = np.sum(dz3, axis=0)  
  
        dz2 = relu_grad(a2) * np.dot(dz3, W3.T)  
        grads['W2'] = np.dot(z1.T, dz2)  
        grads['b2'] = np.sum(dz2, axis=0)  
  
        dz1 = relu_grad(a1) * np.dot(dz2, W2.T)  
        grads['W1'] = np.dot(x.T, dz1)  
        grads['b1'] = np.sum(dz1, axis=0) 
          
        return grads

In [ ]:
# 60 years of data
start_year_train = 1957
start_year_test = 1987
end_year_test = 2016

# Convert the 'Date' column to datetime if it's not already
data['DATE'] = pd.to_datetime(data['DATE'], format='%Y%m')
pred_y_list = []
true_y_list = []

# Loop over the years
for year in tqdm(range(start_year_test, end_year_test+1), desc='Processing', unit='item'):
    # Define the start and end dates for training and test
    start_date_train = pd.Timestamp(year=start_year_train, month=1, day=1)
    end_date_train = pd.Timestamp(year=year-13, month=12, day=31)

    start_date_val = pd.Timestamp(year=year-12, month=1, day=1)
    end_date_val = pd.Timestamp(year=year-1, month=12, day=31)
    
    start_date_test = pd.Timestamp(year=year, month=1, day=1)
    end_date_test = pd.Timestamp(year=year, month=12, day=31)
    

    print(f'start_date_train: {start_date_train.strftime("%Y-%m")}')
    print(f'end_date_train: {end_date_train.strftime("%Y-%m")}')

    print(f'start_date_val: {start_date_val.strftime("%Y-%m")}')
    print(f'end_date_val: {end_date_val.strftime("%Y-%m")}')
    
    print(f'start_date_test: {start_date_test.strftime("%Y-%m")}')
    print(f'end_date_test: {end_date_test.strftime("%Y-%m")}')

    # Select the data for training and test
    train_data = data[(data['DATE'] >= start_date_train) & (data['DATE'] <= end_date_train)]
    val_data = data[(data['DATE'] >= start_date_val) & (data['DATE'] <= end_date_val)]
    test_data = data[(data['DATE'] >= start_date_test) & (data['DATE'] <= end_date_test)]
    
    # Extract features and target variables
    X_train = train_data.drop(['permno', 'DATE', 'RET'], axis=1).values  
    y_train = train_data['RET'].values  
    X_val = val_data.drop(['permno', 'DATE', 'RET'], axis=1).values 
    y_val = val_data['RET'].values
    X_test = test_data.drop(['permno', 'DATE', 'RET'], axis=1).values
    y_test = test_data['RET'].values
    
    network = FourLayerNet(input_size = 94, hidden_size1 = 32, hidden_size2 = 16, hidden_size3 = 8, output_size = 1)  
 
    n_epochs = 100000   
    train_size = x_train.shape[0]
    batch_size = 1000 
    learning_rate = 0.01  

    patience = 20
    best_model_state = None
    best_loss = float('inf')
    counter = 0

    for epoch in range(n_epochs): 
        for i in range(0, train_size, batch_size):
            xi = x_train[i:i + batch_size]
            yi = y_train[i:i + batch_size]  
        
            grad = net_work.gradient(xi, yi)
            for key in ('W1', 'b1', 'W2', 'b2', 'W3', 'b3', 'W4', 'b4'):
                net_work.params[key] -= learning_rate * grad[key]
    
        val_loss = net_work.accuracy(x_val, y_val)
    
        if val_loss < best_loss:
            best_loss = val_loss
            best_model_state = net_work.params.copy()
            counter = 0
        else:
            counter += 1
            if counter >= patience:
                break

        if epoch % 2000 == 0:
            train_loss = net_work.accuracy(x_train, y_train)
            test_loss = net_work.accuracy(x_val, y_val)
            print(f"train loss: {train_loss}, val loss {val_loss}") 
        
    for key in ('W1', 'b1', 'W2', 'b2', 'W3', 'b3', 'W4', 'b4'):
        net_work.params[key] = best_model_state[key]
        
    # Predict on training and validation data
    y_test_pred = model.predict(X_test)

    pred_y_list.extend(y_test_pred)
    true_y_list.extend(y_test)

In [ ]:
R2_cal(np.array(true_y_list), np.array(pred_y_list))* 100

In [ ]:
#五层神经网络  
class FiveLayerNet:  
  
    def __init__(self, input_size, hidden_size1, hidden_size2, hidden_size3, hidden_size4, output_size, weight_init_std=0.01):  
        # 初始化权重和偏置  
        self.params = {}  
        self.params['W1'] = weight_init_std * np.random.randn(input_size, hidden_size1)  
        self.params['b1'] = np.zeros(hidden_size1)  
        self.params['W2'] = weight_init_std * np.random.randn(hidden_size1, hidden_size2)  
        self.params['b2'] = np.zeros(hidden_size2)  
        self.params['W3'] = weight_init_std * np.random.randn(hidden_size2, hidden_size3)  
        self.params['b3'] = np.zeros(hidden_size3)  
        self.params['W4'] = weight_init_std * np.random.randn(hidden_size3, hidden_size4)  
        self.params['b4'] = np.zeros(hidden_size4)  
        self.params['W5'] = weight_init_std * np.random.randn(hidden_size4, output_size)  
        self.params['b5'] = np.zeros(output_size)  
  
    def predict(self, x):  
        W1, W2, W3, W4, W5 = self.params['W1'], self.params['W2'], self.params['W3'], self.params['W4'], self.params['W5']  
        b1, b2, b3, b4, b5 = self.params['b1'], self.params['b2'], self.params['b3'], self.params['b4'], self.params['b5']
          
        a1 = np.dot(x, W1) + b1  
        z1 = relu(a1)  
        a2 = np.dot(z1, W2) + b2  
        z2 = relu(a2)  
        a3 = np.dot(z2, W3) + b3  
        z3 = relu(a3) 
        a4 = np.dot(z3, W4) + b4  
        z4 = relu(a4)
        a5 = np.dot(z4, W5) + b5
        y = a5
          
        return y  
  
    def loss(self, x, t):  
        y = self.predict(x)  
        return mean_squared_error(y, t)  

    def accuracy(self, x, t):
        y = self.predict(x)
        error = np.mean((y - t) ** 2)  
        
        return error
  
    def gradient(self, x, t):  
        grads = {}  
        W1, W2, W3, W4, W5 = self.params['W1'], self.params['W2'], self.params['W3'], self.params['W4'], self.params['W5']
        b1, b2, b3, b4, b5 = self.params['b1'], self.params['b2'], self.params['b3'], self.params['b4'], self.params['b5']
          
        batch_num = x.shape[0]  
          
        a1 = np.dot(x, W1) + b1  
        z1 = relu(a1)  
        a2 = np.dot(z1, W2) + b2  
        z2 = relu(a2)  
        a3 = np.dot(z2, W3) + b3 
        z3 = relu(a3)
        a4 = np.dot(z3, W4) + b4
        z4 = relu(a4)
        a5 = np.dot(z4, W5) + b5
        y = a5
          
        dy = (y - t) / batch_num  
        grads['W5'] = np.dot(z4.T, dy)  
        grads['b5'] = np.sum(dy, axis=0)  
          
        dz4 = relu_grad(a4) * np.dot(dy, W5.T)  
        grads['W4'] = np.dot(z3.T, dz4)  
        grads['b4'] = np.sum(dz4, axis=0)  

        dz3 = relu_grad(a3) * np.dot(dy, W4.T)  
        grads['W3'] = np.dot(z2.T, dz3)  
        grads['b3'] = np.sum(dz3, axis=0)
  
        dz2 = relu_grad(a2) * np.dot(dz3, W3.T)  
        grads['W2'] = np.dot(z1.T, dz2)  
        grads['b2'] = np.sum(dz2, axis=0)  
  
        dz1 = relu_grad(a1) * np.dot(dz2, W2.T)  
        grads['W1'] = np.dot(x.T, dz1)  
        grads['b1'] = np.sum(dz1, axis=0) 
          
        return grads

In [ ]:
# 60 years of data
start_year_train = 1957
start_year_test = 1987
end_year_test = 2016

# Convert the 'Date' column to datetime if it's not already
data['DATE'] = pd.to_datetime(data['DATE'], format='%Y%m')
pred_y_list = []
true_y_list = []

# Loop over the years
for year in tqdm(range(start_year_test, end_year_test+1), desc='Processing', unit='item'):
    # Define the start and end dates for training and test
    start_date_train = pd.Timestamp(year=start_year_train, month=1, day=1)
    end_date_train = pd.Timestamp(year=year-13, month=12, day=31)

    start_date_val = pd.Timestamp(year=year-12, month=1, day=1)
    end_date_val = pd.Timestamp(year=year-1, month=12, day=31)
    
    start_date_test = pd.Timestamp(year=year, month=1, day=1)
    end_date_test = pd.Timestamp(year=year, month=12, day=31)
    

    print(f'start_date_train: {start_date_train.strftime("%Y-%m")}')
    print(f'end_date_train: {end_date_train.strftime("%Y-%m")}')

    print(f'start_date_val: {start_date_val.strftime("%Y-%m")}')
    print(f'end_date_val: {end_date_val.strftime("%Y-%m")}')
    
    print(f'start_date_test: {start_date_test.strftime("%Y-%m")}')
    print(f'end_date_test: {end_date_test.strftime("%Y-%m")}')

    # Select the data for training and test
    train_data = data[(data['DATE'] >= start_date_train) & (data['DATE'] <= end_date_train)]
    val_data = data[(data['DATE'] >= start_date_val) & (data['DATE'] <= end_date_val)]
    test_data = data[(data['DATE'] >= start_date_test) & (data['DATE'] <= end_date_test)]
    
    # Extract features and target variables
    X_train = train_data.drop(['permno', 'DATE', 'RET'], axis=1).values  
    y_train = train_data['RET'].values  
    X_val = val_data.drop(['permno', 'DATE', 'RET'], axis=1).values 
    y_val = val_data['RET'].values
    X_test = test_data.drop(['permno', 'DATE', 'RET'], axis=1).values
    y_test = test_data['RET'].values
    
    network = FiveLayerNet(input_size = 94, hidden_size1 = 32, hidden_size2 = 16, hidden_size3 = 8, hidden_size3 = 4, output_size = 1)  
 
    n_epochs = 100000   
    train_size = x_train.shape[0]
    batch_size = 1000 
    learning_rate = 0.01  

    patience = 20
    best_model_state = None
    best_loss = float('inf')
    counter = 0

    for epoch in range(n_epochs): 
        for i in range(0, train_size, batch_size):
            xi = x_train[i:i + batch_size]
            yi = y_train[i:i + batch_size]  
        
            grad = net_work.gradient(xi, yi)
            for key in ('W1', 'b1', 'W2', 'b2', 'W3', 'b3', 'W4', 'b4', 'W5', 'b5'):
                net_work.params[key] -= learning_rate * grad[key]
    
        val_loss = net_work.accuracy(x_val, y_val)
    
        if val_loss < best_loss:
            best_loss = val_loss
            best_model_state = net_work.params.copy()
            counter = 0
        else:
            counter += 1
            if counter >= patience:
                break

        if epoch % 2000 == 0:
            train_loss = net_work.accuracy(x_train, y_train)
            test_loss = net_work.accuracy(x_val, y_val)
            print(f"train loss: {train_loss}, val loss {val_loss}") 
        
    for key in ('W1', 'b1', 'W2', 'b2', 'W3', 'b3', 'W4', 'b4', 'W5', 'b5'):
        net_work.params[key] = best_model_state[key]
        
    # Predict on training and validation data
    y_test_pred = model.predict(X_test)

    pred_y_list.extend(y_test_pred)
    true_y_list.extend(y_test)

In [ ]:
R2_cal(np.array(true_y_list), np.array(pred_y_list))* 100

In [ ]:
#随机森林模型
# 60 years of data
from sklearn.ensemble import RandomForestRegressor

start_year_train = 1957
start_year_test = 1987
end_year_test = 2016

# Convert the 'Date' column to datetime if it's not already
data['DATE'] = pd.to_datetime(data['DATE'], format='%Y%m')

pred_y_list = []
true_y_list = []

# Loop over the years
for year in tqdm(range(start_year_test, end_year_test+1), desc='Processing', unit='item'):
    # Define the start and end dates for training and test
    start_date_train = pd.Timestamp(year=start_year_train, month=1, day=1)
    end_date_train = pd.Timestamp(year=year-13, month=12, day=31)

    start_date_val = pd.Timestamp(year=year-12, month=1, day=1)
    end_date_val = pd.Timestamp(year=year-1, month=12, day=31)
    
    start_date_test = pd.Timestamp(year=year, month=1, day=1)
    end_date_test = pd.Timestamp(year=year, month=12, day=31)
    

    print(f'start_date_train: {start_date_train.strftime("%Y-%m")}')
    print(f'end_date_train: {end_date_train.strftime("%Y-%m")}')

    print(f'start_date_val: {start_date_val.strftime("%Y-%m")}')
    print(f'end_date_val: {end_date_val.strftime("%Y-%m")}')
    
    print(f'start_date_test: {start_date_test.strftime("%Y-%m")}')
    print(f'end_date_test: {end_date_test.strftime("%Y-%m")}')

    # Select the data for training and test
    train_data = data[(data['DATE'] >= start_date_train) & (data['DATE'] <= end_date_val)]
    test_data = data[(data['DATE'] >= start_date_test) & (data['DATE'] <= end_date_test)]
    
    # Extract features and target variables
    X_train = train_data.drop(['permno', 'DATE', 'RET'], axis=1).values  
    y_train = train_data['RET'].values  
    X_test = test_data.drop(['permno', 'DATE', 'RET'], axis=1).values
    y_test = test_data['RET'].values
  
    model = RandomForestRegressor(random_state = 42)  
    model.fit(X_train, y_train)   
  
    y_test_pred = model.predict(X_test)  

    pred_y_list.extend(y_test_pred)  
    true_y_list.extend(y_test)

In [ ]:
R2_cal(np.array(true_y_list), np.array(pred_y_list))* 100